<a href="https://colab.research.google.com/github/AIML-Dept/RithikaRajavel_1GA23AI041/blob/main/WEEK_5.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Tutorial 5 - Easy: Measurement Probability and Shot Convergence

## Objective

Measure a Hadamard-superposition qubit using different numbers
of shots and observe how the measured probability approaches 0.5.

We will use 100, 1000 and 10000 shots.

According to the Born rule, measuring the |+⟩ state in the
computational basis should produce approximately:

P(0) = 0.5
P(1) = 0.5

In [3]:
!pip install qiskit
!pip install qiskit-aer
from qiskit import QuantumCircuit, transpile
from qiskit_aer import AerSimulator
import pandas as pd

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.4/12.4 MB 96.8 MB/s eta 0:00:00


In [4]:
qc = QuantumCircuit(1, 1)

qc.h(0)
qc.measure(0, 0)

print(qc)

     ┌───┐┌─┐
  q: ┤ H ├┤M├
     └───┘└╥┘
c: 1/══════╩═
           0 


In [5]:
simulator = AerSimulator()

shots_list = [100, 1000, 10000]
results = []

for shots in shots_list:
    compiled = transpile(qc, simulator)
    job = simulator.run(compiled, shots=shots)
    counts = job.result().get_counts()

    probability_0 = counts.get("0", 0) / shots
    probability_1 = counts.get("1", 0) / shots

    results.append([
        shots,
        probability_0,
        probability_1
    ])

df = pd.DataFrame(
    results,
    columns=["Shots", "P(0)", "P(1)"]
)

df

,Shots,P(0),P(1)
0,100,0.5100,0.4900
1,1000,0.5210,0.4790
2,10000,0.4979,0.5021


In [6]:
print(df)

print("\nTheoretical probability:")
print("P(0) = 0.5")
print("P(1) = 0.5")

   Shots    P(0)    P(1)
0    100  0.5100  0.4900
1   1000  0.5210  0.4790
2  10000  0.4979  0.5021

Theoretical probability:
P(0) = 0.5
P(1) = 0.5


In [8]:
from qiskit import QuantumCircuit, transpile
from qiskit_aer import AerSimulator

## Step 1: Prepare the |+⟩ State

The |+⟩ state is created by applying a Hadamard gate
to the initial |0⟩ state.

We then apply another Hadamard gate before measurement
to measure in the X-basis.

In [9]:
plus = QuantumCircuit(1, 1)

plus.h(0)
plus.h(0)
plus.measure(0, 0)

print(plus)

     ┌───┐┌───┐┌─┐
  q: ┤ H ├┤ H ├┤M├
     └───┘└───┘└╥┘
c: 1/═══════════╩═
                0 


In [10]:
minus = QuantumCircuit(1, 1)

minus.x(0)
minus.h(0)
minus.h(0)
minus.measure(0, 0)

print(minus)

     ┌───┐┌───┐┌───┐┌─┐
  q: ┤ X ├┤ H ├┤ H ├┤M├
     └───┘└───┘└───┘└╥┘
c: 1/════════════════╩═
                     0 


In [11]:
simulator = AerSimulator()

plus_job = simulator.run(
    transpile(plus, simulator),
    shots=1024
)

plus_counts = plus_job.result().get_counts()

print("Results for |+⟩:")
print(plus_counts)

Results for |+⟩:
{'0': 1024}


In [12]:
minus_job = simulator.run(
    transpile(minus, simulator),
    shots=1024
)

minus_counts = minus_job.result().get_counts()

print("Results for |-⟩:")
print(minus_counts)

Results for |-⟩:
{'1': 1024}


# Tutorial 5 - Hard: Partial Measurement of a Bell Pair

## Objective

Create a two-qubit Bell pair and measure only one qubit.

We will observe how measuring one qubit affects the state
of the remaining unmeasured qubit.

This demonstrates the collapse of an entangled quantum state.

In [13]:
from qiskit import QuantumCircuit
from qiskit.quantum_info import Statevector

In [14]:
qc = QuantumCircuit(2, 1)

qc.h(0)
qc.cx(0, 1)

print(qc)

     ┌───┐     
q_0: ┤ H ├──■──
     └───┘┌─┴─┐
q_1: ─────┤ X ├
          └───┘
c: 1/══════════
               


In [15]:
state = Statevector.from_instruction(qc)

print("Initial Bell State:")
print(state)

Initial Bell State:
Statevector([0.70710678+0.j, 0.        +0.j, 0.        +0.j,
             0.70710678+0.j],
            dims=(2, 2))


In [16]:
qc.measure(0, 0)

print(qc)

     ┌───┐     ┌─┐
q_0: ┤ H ├──■──┤M├
     └───┘┌─┴─┐└╥┘
q_1: ─────┤ X ├─╫─
          └───┘ ║ 
c: 1/═══════════╩═
                0 


In [17]:
from qiskit import transpile
from qiskit_aer import AerSimulator

simulator = AerSimulator()

job = simulator.run(
    transpile(qc, simulator),
    shots=1024
)

counts = job.result().get_counts()

print("Measurement results:")
print(counts)

Measurement results:
{'1': 510, '0': 514}


# Tutorial 5 - Real World: Estimating an Unknown Rotation Angle

## Objective

Estimate an unknown rotation angle applied to a qubit
using repeated measurement statistics.

A rotation around the Y-axis changes the probability
of measuring |0⟩.

We will use the observed probability to estimate the
rotation angle.

In [18]:
import numpy as np
from qiskit import QuantumCircuit, transpile
from qiskit_aer import AerSimulator

In [19]:
true_angle = np.pi / 3

qc = QuantumCircuit(1, 1)

qc.ry(true_angle, 0)
qc.measure(0, 0)

print(qc)

     ┌─────────┐┌─┐
  q: ┤ Ry(π/3) ├┤M├
     └─────────┘└╥┘
c: 1/════════════╩═
                 0 


In [20]:
simulator = AerSimulator()

shots = 10000

job = simulator.run(
    transpile(qc, simulator),
    shots=shots
)

counts = job.result().get_counts()

p0 = counts.get("0", 0) / shots

print("Measurement counts:", counts)
print(f"Estimated P(0): {p0:.4f}")

Measurement counts: {'1': 2578, '0': 7422}
Estimated P(0): 0.7422


In [21]:
estimated_angle = 2 * np.arccos(np.sqrt(p0))

print(f"Estimated angle: {estimated_angle:.4f} radians")
print(f"Actual angle:    {true_angle:.4f} radians")

Estimated angle: 1.0651 radians
Actual angle:    1.0472 radians


# Tutorial 5 - Challenge: Quantum Randomness vs Classical Bias

## Objective

Design an experiment to distinguish a genuinely random
quantum bit generator from a biased classical
pseudo-random generator.

A Hadamard gate followed by measurement produces
approximately equal numbers of 0 and 1.

We compare this with a deliberately biased classical
generator.

A chi-square test is used to measure how strongly the
observed data differs from a 50/50 distribution.

In [22]:
import random
import numpy as np
from qiskit import QuantumCircuit, transpile
from qiskit_aer import AerSimulator
from scipy.stats import chisquare

In [23]:
shots = 10000

qc = QuantumCircuit(1, 1)

qc.h(0)
qc.measure(0, 0)

simulator = AerSimulator()

job = simulator.run(
    transpile(qc, simulator),
    shots=shots
)

quantum_counts = job.result().get_counts()

quantum_0 = quantum_counts.get("0", 0)
quantum_1 = quantum_counts.get("1", 0)

print("Quantum counts:")
print(quantum_counts)

Quantum counts:
{'0': 4980, '1': 5020}


In [24]:
classical_bits = [
    0 if random.random() < 0.70 else 1
    for _ in range(shots)
]

classical_0 = classical_bits.count(0)
classical_1 = classical_bits.count(1)

print("Classical counts:")
print({
    "0": classical_0,
    "1": classical_1
})

Classical counts:
{'0': 7011, '1': 2989}


In [25]:
quantum_test = chisquare(
    [quantum_0, quantum_1],
    f_exp=[shots / 2, shots / 2]
)

classical_test = chisquare(
    [classical_0, classical_1],
    f_exp=[shots / 2, shots / 2]
)

print("Quantum generator:")
print(f"Chi-square statistic = {quantum_test.statistic:.3f}")
print(f"p-value = {quantum_test.pvalue:.6f}")

print("\nClassical generator:")
print(f"Chi-square statistic = {classical_test.statistic:.3f}")
print(f"p-value = {classical_test.pvalue:.6f}")

Quantum generator:
Chi-square statistic = 0.160
p-value = 0.689157

Classical generator:
Chi-square statistic = 1617.648
p-value = 0.000000


## Conclusion

The experiment compared measurement results from a
quantum random bit generator with a deliberately biased
classical generator.

Statistical hypothesis testing provides a way to identify
whether the observed distribution is consistent with
genuine 50/50 randomness.

This demonstrates how statistical methods can be used to
analyse quantum random number generation.